# **EN3160 Image Processing & Computer Vision - Assignment 03**


**Name : Meedeniya M.M.H**  
**Index No. : 230407K**

### **Github Repository** : https://github.com/manjarimeedeniya/EN3160-Assignment3

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt


### 2(a)

In [ ]:
# Device configuration & seed
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# LeNet-5 needs 32x32: Pad MNIST (28x28) with 2 pixels on each side
transform = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

batch_size = 64
epochs = 10
learning_rate = 0.001

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.features = nn.Sequential(
            # C1: Input (1, 32, 32) -> Output (6, 28, 28)
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1),
            nn.Tanh(),
            # S2: Output (6, 14, 14)
            nn.AvgPool2d(kernel_size=2, stride=2),
            # C3: Output (16, 10, 10)
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1),
            nn.Tanh(),
            # S4: Output (16, 5, 5)
            nn.AvgPool2d(kernel_size=2, stride=2),
            # C5: Output (120, 1, 1)
            nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5, stride=1),
            nn.Tanh()
        )
        self.classifier = nn.Sequential(
            # F6: Linear mapping from 120 -> 84
            nn.Linear(120, 84),
            nn.Tanh(),
            # Output: 84 -> 10 classes
            nn.Linear(84, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)  # Flatten (batch_size, 120, 1, 1) to (batch_size, 120)
        logits = self.classifier(x)
        return logits

model = LeNet5().to(device)
print(model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

train_losses, test_losses = [], []
train_accuracies, test_accuracies = []

for epoch in range(epochs):
    # --- Training Phase ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    for inputs, targets in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        total_train += targets.size(0)
        correct_train += (preds == targets).sum().item()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = 100.0 * correct_train / total_train

    # --- Testing Phase ---
    model.eval()
    test_loss, correct_test, total_test = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            total_test += targets.size(0)
            correct_test += (preds == targets).sum().item()

    epoch_test_loss = test_loss / total_test
    epoch_test_acc = 100.0 * correct_test / total_test

    train_losses.append(epoch_train_loss)
    test_losses.append(epoch_test_loss)
    train_accuracies.append(epoch_train_acc)
    test_accuracies.append(epoch_test_acc)

    print(f"Epoch {epoch+1:02d}/10 | "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
          f"Test Loss: {epoch_test_loss:.4f} | Test Acc: {epoch_test_acc:.2f}%")

print(f"\nFinal Result after 10 epochs:")
print(f"Training Accuracy: {train_accuracies[-1]:.2f}%")
print(f"Test Accuracy:     {test_accuracies[-1]:.2f}%")